# STEP 4. 모델 실행
네눈쑥가지나방 위험도 예측
- Sine Curve 적산온도 계산 → 누적 DD → Sigmoid → 위험도 등급
- 파라미터 출처: Choi & Kim 2014 (Crop Protection 66:72-79)
- ⚠ b=20, 2화기 X_Value=745.0 은 임의값 → 전문가 자문 후 보정 필요

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

SAVE_DIR  = '/content/drive/MyDrive/JADX_병해충/data'
SAVE_FILE = f'{SAVE_DIR}/tb_weather_pest.csv'

df = pd.read_csv(SAVE_FILE)
df['date']  = pd.to_datetime(df['crtr_ymd'], format='%Y%m%d')
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['jld']   = df['date'].dt.dayofyear

print(f'✅ 기상 데이터 로드: {len(df)}건')
print(df['stn_nm'].unique())

Mounted at /content/drive
✅ 기상 데이터 로드: 8768건
['제주' '고산' '성산' '서귀포']


In [2]:
# ── 파라미터 설정 ──────────────────────────────────
INSECT_NAME = '네눈쑥가지나방'

# 발육 온도 파라미터 (Choi & Kim 2014, 번데기 기준)
T_LOW = 9.8
T_OPT = 31.0
T_UPP = 37.8

# 세대별 Sigmoid 파라미터
# X_Value: 세대 피크 DD / Y_Value: b값 (⚠ 임의값, 자문 필요)
GENERATION_CONFIG = [
    {'gen': '1화기', 'X_Value': 188.7, 'Y_Value': 20},
    {'gen': '2화기', 'X_Value': 745.0, 'Y_Value': 20},
]

# 위험도 임계값 (엑셀 모델 기준)
RISK_THRESHOLDS = {
    '1화기': {'주의': 188.7, '경보': 240.0, '심각': 289.4},
    '2화기': {'주의': 745.0, '경보': 883.7, '심각': 984.4},
}

print('✅ 파라미터 설정 완료')
for g in GENERATION_CONFIG:
    print(f"  {g['gen']}: X={g['X_Value']} DD, b={g['Y_Value']}")

✅ 파라미터 설정 완료
  1화기: X=188.7 DD, b=20
  2화기: X=745.0 DD, b=20


In [3]:
# ── Sine Curve DD 계산 함수 ────────────────────────
def sine_dd(tmax, tmin):
    sub1 = (tmax - tmin) / 2
    sub2 = (tmax + tmin) / 2
    if abs(sub1) < 1e-9:
        return 0
    q_low  = np.arcsin(np.clip((T_LOW - sub2) / sub1, -1, 1))
    q_high = np.arcsin(np.clip((T_OPT - sub2) / sub1, -1, 1))
    sub3 = sub2 - T_LOW if tmin > T_LOW else \
           (1/np.pi)*((sub2-T_LOW)*(np.pi/2-q_low)+sub1*np.cos(q_low))
    sub4 = (1/np.pi)*((sub2-T_LOW)*(q_high+np.pi/2)+(T_OPT-T_LOW)*(np.pi/2-q_high)-sub1*np.cos(q_high))
    sub5 = (1/np.pi)*((sub2-T_LOW)*(q_high-q_low)+sub1*(np.cos(q_high)-np.cos(q_low))+(T_OPT-T_LOW)*(np.pi/2-q_high))
    if tmax > T_UPP or tmax < T_LOW: return 0
    elif tmin > T_OPT:               return T_OPT - T_LOW
    elif tmax < T_OPT:               return sub3
    else:                            return sub4 if tmin > T_LOW else sub5

def sigmoid(dd, x_val, y_val):
    return 1 / (1 + np.exp(-(dd - x_val) / y_val))

def get_risk_grade(cumdd):
    for gen, thr in RISK_THRESHOLDS.items():
        if cumdd >= thr['심각']: return f'{gen} 심각'
        if cumdd >= thr['경보']: return f'{gen} 경보'
        if cumdd >= thr['주의']: return f'{gen} 주의'
    return '보통'

print('✅ 함수 정의 완료')

✅ 함수 정의 완료


In [4]:
# ── 모델 실행 ──────────────────────────────────────
results = []

for (stn, year), grp in df.groupby(['stn_nm', 'year']):
    grp = grp.sort_values('jld').reset_index(drop=True)

    # daily_dd 계산
    grp['daily_dd'] = grp.apply(
        lambda r: sine_dd(r['day_hghst_tp'], r['day_lowst_tp']), axis=1
    )

    # 연간 누적 DD (1월1일 기산)
    grp['cumdd'] = grp['daily_dd'].cumsum()

    # Sigmoid 세대별 개체군
    for g in GENERATION_CONFIG:
        grp[f"sigmoid_{g['gen']}"] = grp['cumdd'].apply(
            lambda x: sigmoid(x, g['X_Value'], g['Y_Value'])
        )

    # 위험도 등급
    grp['risk_grade'] = grp['cumdd'].apply(get_risk_grade)
    grp['stn_nm']     = stn

    results.append(grp)

result_df = pd.concat(results, ignore_index=True)

# 저장
RESULT_FILE = f'{SAVE_DIR}/result_pest_model.csv'
result_df.to_csv(RESULT_FILE, index=False, encoding='utf-8-sig')

print(f'✅ 모델 실행 완료: {len(result_df)}건')
print(f'   저장: {RESULT_FILE}')

✅ 모델 실행 완료: 8768건
   저장: /content/drive/MyDrive/JADX_병해충/data/result_pest_model.csv


In [5]:
# ── 위험도 발생일 요약 ─────────────────────────────
print('='*60)
print(f'  [{INSECT_NAME}] 위험도 발생일 요약')
print('='*60)

for stn in result_df['stn_nm'].unique():
    print(f'\n  ▶ {stn}')
    for year in sorted(result_df['year'].unique()):
        grp = result_df[(result_df['stn_nm']==stn) & (result_df['year']==year)]
        row_str = f'    {year}년: '
        for gen, thr in RISK_THRESHOLDS.items():
            hit = grp[grp['cumdd'] >= thr['주의']]
            if len(hit) > 0:
                r = hit.iloc[0]
                row_str += f"{gen} 주의={r['date'].strftime('%m/%d')}(J{int(r['jld'])})  "
            else:
                row_str += f'{gen} 주의=미도달  '
        print(row_str)

  [네눈쑥가지나방] 위험도 발생일 요약

  ▶ 고산
    2020년: 1화기 주의=04/28(J119)  2화기 주의=06/25(J177)  
    2021년: 1화기 주의=04/06(J96)  2화기 주의=06/13(J164)  
    2022년: 1화기 주의=04/23(J113)  2화기 주의=06/20(J171)  
    2023년: 1화기 주의=04/18(J108)  2화기 주의=06/21(J172)  
    2024년: 1화기 주의=04/14(J105)  2화기 주의=06/18(J170)  
    2025년: 1화기 주의=04/27(J117)  2화기 주의=06/29(J180)  

  ▶ 서귀포
    2020년: 1화기 주의=04/04(J95)  2화기 주의=06/13(J165)  
    2021년: 1화기 주의=03/29(J88)  2화기 주의=06/05(J156)  
    2022년: 1화기 주의=04/12(J102)  2화기 주의=06/11(J162)  
    2023년: 1화기 주의=04/01(J91)  2화기 주의=06/09(J160)  
    2024년: 1화기 주의=04/07(J98)  2화기 주의=06/07(J159)  
    2025년: 1화기 주의=04/16(J106)  2화기 주의=06/15(J166)  

  ▶ 성산
    2020년: 1화기 주의=04/20(J111)  2화기 주의=06/22(J174)  
    2021년: 1화기 주의=04/06(J96)  2화기 주의=06/13(J164)  
    2022년: 1화기 주의=04/21(J111)  2화기 주의=06/20(J171)  
    2023년: 1화기 주의=04/11(J101)  2화기 주의=06/16(J167)  
    2024년: 1화기 주의=04/13(J104)  2화기 주의=06/15(J167)  
    2025년: 1화기 주의=04/20(J110)  2화기 주의=06/23(J174)  

  ▶ 제주
    2020년: 1화기

In [6]:
# ── 논문 기준값과 비교 (검증) ──────────────────────
print('='*60)
print('  [검증] 논문 기준값 비교 (Choi et al. 2011)')
print('  기준: 1화기 성충 피크 Julian 139~145 (5월 중순)')
print('  → 예보는 피크 2주 전 Julian 120~135 가 적정')
print('='*60)

TARGET_LOW, TARGET_HIGH = 120, 135  # 권장 Julian 범위

for stn in result_df['stn_nm'].unique():
    issues = []
    for year in sorted(result_df['year'].unique()):
        grp = result_df[(result_df['stn_nm']==stn) & (result_df['year']==year)]
        hit = grp[grp['cumdd'] >= RISK_THRESHOLDS['1화기']['주의']]
        if len(hit) > 0:
            jld = int(hit.iloc[0]['jld'])
            status = '✅' if TARGET_LOW <= jld <= TARGET_HIGH else '⚠️'
            if status == '⚠️':
                diff = jld - TARGET_LOW if jld < TARGET_LOW else jld - TARGET_HIGH
                issues.append(f'{year}년 J{jld} ({diff:+d}일)')
            print(f'  {status} {stn} {year}년: Julian {jld} ({"범위 내" if status=="✅" else "범위 벗어남"})')

print(f'\n  → 범위 벗어난 경우: {len(issues)}건')
if issues:
    print(f'  → {issues}')
    print('  → 임계값 보정 필요 여부 교수 자문 필요')

  [검증] 논문 기준값 비교 (Choi et al. 2011)
  기준: 1화기 성충 피크 Julian 139~145 (5월 중순)
  → 예보는 피크 2주 전 Julian 120~135 가 적정
  ⚠️ 고산 2020년: Julian 119 (범위 벗어남)
  ⚠️ 고산 2021년: Julian 96 (범위 벗어남)
  ⚠️ 고산 2022년: Julian 113 (범위 벗어남)
  ⚠️ 고산 2023년: Julian 108 (범위 벗어남)
  ⚠️ 고산 2024년: Julian 105 (범위 벗어남)
  ⚠️ 고산 2025년: Julian 117 (범위 벗어남)
  ⚠️ 서귀포 2020년: Julian 95 (범위 벗어남)
  ⚠️ 서귀포 2021년: Julian 88 (범위 벗어남)
  ⚠️ 서귀포 2022년: Julian 102 (범위 벗어남)
  ⚠️ 서귀포 2023년: Julian 91 (범위 벗어남)
  ⚠️ 서귀포 2024년: Julian 98 (범위 벗어남)
  ⚠️ 서귀포 2025년: Julian 106 (범위 벗어남)
  ⚠️ 성산 2020년: Julian 111 (범위 벗어남)
  ⚠️ 성산 2021년: Julian 96 (범위 벗어남)
  ⚠️ 성산 2022년: Julian 111 (범위 벗어남)
  ⚠️ 성산 2023년: Julian 101 (범위 벗어남)
  ⚠️ 성산 2024년: Julian 104 (범위 벗어남)
  ⚠️ 성산 2025년: Julian 110 (범위 벗어남)
  ⚠️ 제주 2020년: Julian 107 (범위 벗어남)
  ⚠️ 제주 2021년: Julian 92 (범위 벗어남)
  ⚠️ 제주 2022년: Julian 109 (범위 벗어남)
  ⚠️ 제주 2023년: Julian 99 (범위 벗어남)
  ⚠️ 제주 2024년: Julian 101 (범위 벗어남)
  ⚠️ 제주 2025년: Julian 108 (범위 벗어남)

  → 범위 벗어난 경우: 6건
  → ['2020년 J107 (-13일)', '2021년